In [6]:
import os

os.makedirs("templates", exist_ok=True)
print("Created 'templates' folder successfully.")

Created 'templates' folder successfully.


In [7]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>Mealy to Moore FSM Converter</title>
  <script src="https://cdn.tailwindcss.com"></script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/viz.js/2.1.2/viz.js"></script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/viz.js/2.1.2/full.render.js"></script>
  <style>
    body { font-family: 'Inter', system-ui, sans-serif; background-color: #f8fafc; }
    .diagram-box svg { max-width: 100%; height: auto; margin: 0 auto; }
  </style>
</head>
<body class="text-slate-800">

  <!-- Header -->
  <header class="bg-indigo-700 text-white shadow-md">
    <div class="max-w-7xl mx-auto px-6 py-4 flex flex-wrap items-center justify-between gap-4">
      <div>
        <h1 class="text-2xl font-bold tracking-tight">Mealy → Moore Machine Converter</h1>
        <p class="text-indigo-200 text-sm">Automata Theory State-Splitting Engine, Visualizer & Equivalence Simulator</p>
      </div>
      <div class="flex gap-2">
        <button onclick="loadPreset('twos_comp')" class="bg-indigo-600 hover:bg-indigo-500 border border-indigo-400 px-3 py-1.5 rounded text-sm font-medium transition">
          Preset: 2's Complement
        </button>
        <button onclick="loadPreset('seq_101')" class="bg-indigo-600 hover:bg-indigo-500 border border-indigo-400 px-3 py-1.5 rounded text-sm font-medium transition">
          Preset: Detect "101"
        </button>
      </div>
    </div>
  </header>

  <main class="max-w-7xl mx-auto px-6 py-8 space-y-8">

    <!-- Error Banner -->
    <div id="errorBanner" class="hidden bg-red-50 border-l-4 border-red-500 text-red-700 p-4 rounded shadow-sm"></div>

    <!-- Step 1: Machine Definition -->
    <section class="bg-white p-6 rounded-xl shadow-sm border border-slate-200">
      <h2 class="text-lg font-semibold text-slate-900 mb-4">1. Define Mealy Machine Tuple (Q, Σ, Δ, δ, λ, q<sub>0</sub>)</h2>
      <div class="grid grid-cols-1 md:grid-cols-4 gap-4">
        <div>
          <label class="block text-xs font-semibold uppercase text-slate-500 mb-1">States (Q, comma-separated)</label>
          <input id="statesInput" type="text" value="q0, q1" class="w-full border border-slate-300 rounded-lg px-3 py-2 text-sm focus:ring-2 focus:ring-indigo-500 outline-none" />
        </div>
        <div>
          <label class="block text-xs font-semibold uppercase text-slate-500 mb-1">Input Alphabet (Σ)</label>
          <input id="inputsInput" type="text" value="0, 1" class="w-full border border-slate-300 rounded-lg px-3 py-2 text-sm focus:ring-2 focus:ring-indigo-500 outline-none" />
        </div>
        <div>
          <label class="block text-xs font-semibold uppercase text-slate-500 mb-1">Output Alphabet (Δ)</label>
          <input id="outputsInput" type="text" value="0, 1" class="w-full border border-slate-300 rounded-lg px-3 py-2 text-sm focus:ring-2 focus:ring-indigo-500 outline-none" />
        </div>
        <div>
          <label class="block text-xs font-semibold uppercase text-slate-500 mb-1">Initial State (q<sub>0</sub>)</label>
          <input id="initialInput" type="text" value="q0" class="w-full border border-slate-300 rounded-lg px-3 py-2 text-sm focus:ring-2 focus:ring-indigo-500 outline-none" />
        </div>
      </div>
      <div class="mt-4 flex justify-end">
        <button onclick="buildTransitionTable()" class="bg-slate-800 hover:bg-slate-700 text-white px-4 py-2 rounded-lg text-sm font-medium transition">
          Generate Transition Table
        </button>
      </div>
    </section>

    <!-- Step 2: Mealy Transition Matrix -->
    <section class="bg-white p-6 rounded-xl shadow-sm border border-slate-200">
      <div class="flex items-center justify-between mb-4">
        <h2 class="text-lg font-semibold text-slate-900">2. Configure Mealy Transition Table</h2>
        <button onclick="convertMachine()" class="bg-indigo-600 hover:bg-indigo-700 text-white px-5 py-2 rounded-lg text-sm font-semibold shadow-sm transition">
          Convert to Moore Machine →
        </button>
      </div>
      <div id="mealyTableContainer" class="overflow-x-auto"></div>
    </section>

    <!-- Step 3: Conversion Results -->
    <div id="resultsSection" class="hidden space-y-8">

      <!-- Comparison Tables -->
      <section class="grid grid-cols-1 lg:grid-cols-2 gap-6">
        <!-- Original Mealy Summary -->
        <div class="bg-white p-6 rounded-xl shadow-sm border border-slate-200">
          <h3 class="text-md font-bold text-indigo-700 mb-3">Original Mealy Transition Table</h3>
          <div id="resultMealyTable" class="overflow-x-auto"></div>
        </div>

        <!-- Converted Moore Table -->
        <div class="bg-white p-6 rounded-xl shadow-sm border border-emerald-200">
          <div class="flex items-center justify-between mb-3">
            <h3 class="text-md font-bold text-emerald-700">Converted Moore Transition Table</h3>
            <span id="mooreInitBadge" class="text-xs bg-emerald-100 text-emerald-800 px-2.5 py-1 rounded-full font-semibold"></span>
          </div>
          <div id="resultMooreTable" class="overflow-x-auto"></div>
        </div>
      </section>

      <!-- State Diagrams -->
      <section class="grid grid-cols-1 lg:grid-cols-2 gap-6">
        <div class="bg-white p-6 rounded-xl shadow-sm border border-slate-200">
          <h3 class="text-md font-bold text-indigo-700 mb-3">Mealy State Diagram (Input / Output)</h3>
          <div id="mealyGraph" class="diagram-box min-h-[220px] flex items-center justify-center bg-slate-50 rounded-lg p-4 border border-slate-100"></div>
        </div>
        <div class="bg-white p-6 rounded-xl shadow-sm border border-emerald-200">
          <h3 class="text-md font-bold text-emerald-700 mb-3">Moore State Diagram (State / Output)</h3>
          <div id="mooreGraph" class="diagram-box min-h-[220px] flex items-center justify-center bg-slate-50 rounded-lg p-4 border border-slate-100"></div>
        </div>
      </section>

      <!-- Step 4: Equivalence Simulator -->
      <section class="bg-white p-6 rounded-xl shadow-sm border border-slate-200">
        <h3 class="text-lg font-semibold text-slate-900 mb-2">3. Input Sequence Equivalence Simulator</h3>
        <p class="text-sm text-slate-500 mb-4">Test an input sequence (space or comma separated, or continuous characters if single-char symbols) on both machines.</p>
        <div class="flex flex-wrap gap-3 items-center">
          <input id="simSequenceInput" type="text" placeholder="e.g. 10110" value="10110" class="border border-slate-300 rounded-lg px-4 py-2 text-sm w-64 focus:ring-2 focus:ring-indigo-500 outline-none" />
          <button onclick="runSimulation()" class="bg-emerald-600 hover:bg-emerald-700 text-white px-5 py-2 rounded-lg text-sm font-semibold transition">
            Run Simulation
          </button>
        </div>
        <div id="simOutputBox" class="mt-4 hidden"></div>
      </section>

    </div>
  </main>

  <script>
    let currentConversionData = null;
    const viz = new Viz();

    function parseCSV(str) {
      return str.split(',').map(s => s.trim()).filter(Boolean);
    }

    function showError(msg) {
      const el = document.getElementById('errorBanner');
      el.textContent = msg;
      el.classList.remove('hidden');
    }

    function hideError() {
      document.getElementById('errorBanner').classList.add('hidden');
    }

    function buildTransitionTable(presetTransitions = null) {
      hideError();
      const states = parseCSV(document.getElementById('statesInput').value);
      const inputs = parseCSV(document.getElementById('inputsInput').value);
      const outputs = parseCSV(document.getElementById('outputsInput').value);

      if (!states.length || !inputs.length || !outputs.length) {
        showError("Please enter valid comma-separated lists for states, inputs, and outputs.");
        return;
      }

      let html = `<table class="min-w-full border-collapse border border-slate-300 text-sm">
        <thead>
          <tr class="bg-slate-100">
            <th rowspan="2" class="border border-slate-300 px-4 py-2 text-left">Present State</th>
            ${inputs.map(inp => `<th colspan="2" class="border border-slate-300 px-4 py-2 text-center bg-indigo-50 text-indigo-900">Input = ${inp}</th>`).join('')}
          </tr>
          <tr class="bg-slate-50 text-xs text-slate-600">
            ${inputs.map(() => `<th class="border border-slate-300 px-3 py-1.5">Next State</th><th class="border border-slate-300 px-3 py-1.5">Output</th>`).join('')}
          </tr>
        </thead>
        <tbody>`;

      states.forEach(st => {
        html += `<tr>
          <td class="border border-slate-300 px-4 py-2 font-semibold bg-slate-50">${st}</td>`;
        inputs.forEach(inp => {
          const defNext = presetTransitions?.[st]?.[inp]?.next_state || states[0];
          const defOut = presetTransitions?.[st]?.[inp]?.output || outputs[0];

          const stateOpts = states.map(s => `<option value="${s}" ${s === defNext ? 'selected' : ''}>${s}</option>`).join('');
          const outOpts = outputs.map(o => `<option value="${o}" ${o === defOut ? 'selected' : ''}>${o}</option>`).join('');

          html += `
            <td class="border border-slate-300 px-2 py-1.5">
              <select id="next_${st}_${inp}" class="w-full border border-slate-200 rounded px-2 py-1 bg-white">${stateOpts}</select>
            </td>
            <td class="border border-slate-300 px-2 py-1.5">
              <select id="out_${st}_${inp}" class="w-full border border-slate-200 rounded px-2 py-1 bg-white">${outOpts}</select>
            </td>`;
        });
        html += `</tr>`;
      });

      html += `</tbody></table>`;
      document.getElementById('mealyTableContainer').innerHTML = html;
    }

    async function convertMachine() {
      hideError();
      const states = parseCSV(document.getElementById('statesInput').value);
      const inputs = parseCSV(document.getElementById('inputsInput').value);
      const outputs = parseCSV(document.getElementById('outputsInput').value);
      const initial_state = document.getElementById('initialInput').value.trim();

      const transitions = {};
      for (const st of states) {
        transitions[st] = {};
        for (const inp of inputs) {
          const nextEl = document.getElementById(`next_${st}_${inp}`);
          const outEl = document.getElementById(`out_${st}_${inp}`);
          if (!nextEl || !outEl) {
            showError("Please click 'Generate Transition Table' first.");
            return;
          }
          transitions[st][inp] = {
            next_state: nextEl.value,
            output: outEl.value
          };
        }
      }

      try {
        const res = await fetch('api/convert', {
          method: 'POST',
          headers: { 'Content-Type': 'application/json' },
          body: JSON.stringify({ states, inputs, outputs, initial_state, transitions })
        });
        const json = await res.json();
        if (!json.success) {
          showError(json.error);
          return;
        }
        currentConversionData = json.data;
        renderResults(json.data);
      } catch (err) {
        showError("Network error communicating with backend: " + err.message);
      }
    }

    function renderResults(data) {
      document.getElementById('resultsSection').classList.remove('hidden');

      const m = data.mealy;
      let mealyHtml = `<table class="min-w-full border-collapse border border-slate-300 text-sm">
        <thead>
          <tr class="bg-indigo-50">
            <th class="border border-slate-300 px-3 py-2 text-left">State</th>
            ${m.inputs.map(i => `<th class="border border-slate-300 px-3 py-2 text-center">Input ${i} (Next, Out)</th>`).join('')}
          </tr>
        </thead><tbody>`;
      m.states.forEach(st => {
        const initTag = st === m.initial_state ? ' <span class="text-xs text-indigo-600 font-bold">(Start)</span>' : '';
        mealyHtml += `<tr>
          <td class="border border-slate-300 px-3 py-2 font-medium">${st}${initTag}</td>
          ${m.inputs.map(i => {
            const t = m.transitions[st][i];
            return `<td class="border border-slate-300 px-3 py-2 text-center font-mono">${t.next_state}, <span class="text-indigo-600 font-bold">${t.output}</span></td>`;
          }).join('')}
        </tr>`;
      });
      mealyHtml += `</tbody></table>`;
      document.getElementById('resultMealyTable').innerHTML = mealyHtml;

      const mo = data.moore;
      document.getElementById('mooreInitBadge').textContent = `Initial State: ${mo.initial_state}`;
      let mooreHtml = `<table class="min-w-full border-collapse border border-slate-300 text-sm">
        <thead>
          <tr class="bg-emerald-50">
            <th class="border border-slate-300 px-3 py-2 text-left">Moore State</th>
            ${mo.inputs.map(i => `<th class="border border-slate-300 px-3 py-2 text-center">Next (Input ${i})</th>`).join('')}
            <th class="border border-slate-300 px-3 py-2 text-center bg-emerald-100">State Output</th>
          </tr>
        </thead><tbody>`;
      mo.states.forEach(st => {
        const initTag = st === mo.initial_state ? ' <span class="text-xs text-emerald-700 font-bold">(Start)</span>' : '';
        mooreHtml += `<tr>
          <td class="border border-slate-300 px-3 py-2 font-medium">${st}${initTag}</td>
          ${mo.inputs.map(i => `<td class="border border-slate-300 px-3 py-2 text-center font-mono">${mo.transitions[st][i]}</td>`).join('')}
          <td class="border border-slate-300 px-3 py-2 text-center font-mono font-bold text-emerald-700 bg-emerald-50/50">${mo.state_outputs[st]}</td>
        </tr>`;
      });
      mooreHtml += `</tbody></table>`;
      document.getElementById('resultMooreTable').innerHTML = mooreHtml;

      viz.renderSVGElement(m.dot).then(svg => {
        const box = document.getElementById('mealyGraph');
        box.innerHTML = '';
        box.appendChild(svg);
      });
      viz.renderSVGElement(mo.dot).then(svg => {
        const box = document.getElementById('mooreGraph');
        box.innerHTML = '';
        box.appendChild(svg);
      });
    }

    async function runSimulation() {
      if (!currentConversionData) return;
      hideError();

      const raw = document.getElementById('simSequenceInput').value.trim();
      let sequence = [];

      if (raw.includes(',') || raw.includes(' ')) {
        sequence = raw.split(/[\s,]+/).filter(Boolean);
      } else {
        sequence = raw.split('');
      }

      try {
        const res = await fetch('api/simulate', {
          method: 'POST',
          headers: { 'Content-Type': 'application/json' },
          body: JSON.stringify({
            sequence,
            mealy: currentConversionData.mealy,
            moore: currentConversionData.moore
          })
        });
        const json = await res.json();
        if (!json.success) {
          showError(json.error);
          return;
        }

        const outBox = document.getElementById('simOutputBox');
        outBox.classList.remove('hidden');
        outBox.innerHTML = `
          <div class="p-4 rounded-lg border ${json.equivalent ? 'bg-emerald-50 border-emerald-200' : 'bg-amber-50 border-amber-200'} space-y-2 text-sm">
            <div class="font-bold ${json.equivalent ? 'text-emerald-800' : 'text-amber-800'}">
              ${json.equivalent ? 'Verified Equivalent Output Sequence!' : 'Output Mismatch'}
            </div>
            <div><span class="font-semibold">Input Sequence:</span> <code class="bg-white px-2 py-0.5 rounded border">${sequence.join(' ')}</code></div>
            <div><span class="font-semibold">Mealy Output:</span> <code class="bg-white px-2 py-0.5 rounded border text-indigo-700 font-bold">${json.mealy_output}</code></div>
            <div><span class="font-semibold">Moore Output:</span> <code class="bg-white px-2 py-0.5 rounded border text-emerald-700 font-bold">[${json.moore_initial_output}] ${json.moore_output}</code> <span class="text-xs text-slate-500">(Initial state output in brackets, followed by transition outputs)</span></div>
          </div>
        `;
      } catch (err) {
        showError("Simulation failed: " + err.message);
      }
    }

    function loadPreset(type) {
      if (type === 'twos_comp') {
        document.getElementById('statesInput').value = 'q0, q1';
        document.getElementById('inputsInput').value = '0, 1';
        document.getElementById('outputsInput').value = '0, 1';
        document.getElementById('initialInput').value = 'q0';
        const transitions = {
          q0: { '0': { next_state: 'q0', output: '0' }, '1': { next_state: 'q1', output: '1' } },
          q1: { '0': { next_state: 'q1', output: '1' }, '1': { next_state: 'q1', output: '0' } }
        };
        buildTransitionTable(transitions);
        convertMachine();
      } else if (type === 'seq_101') {
        document.getElementById('statesInput').value = 'A, B, C';
        document.getElementById('inputsInput').value = '0, 1';
        document.getElementById('outputsInput').value = '0, 1';
        document.getElementById('initialInput').value = 'A';
        const transitions = {
          A: { '0': { next_state: 'A', output: '0' }, '1': { next_state: 'B', output: '0' } },
          B: { '0': { next_state: 'C', output: '0' }, '1': { next_state: 'B', output: '0' } },
          C: { '0': { next_state: 'A', output: '0' }, '1': { next_state: 'B', output: '1' } }
        };
        buildTransitionTable(transitions);
        convertMachine();
      }
    }

    window.onload = () => loadPreset('twos_comp');
  </script>
</body>
</html>

Overwriting templates/index.html


In [8]:
%%writefile app.py
from flask import Flask, render_template, request, jsonify
from collections import defaultdict

app = Flask(__name__)


class MealyToMooreConverter:
    def __init__(self, states, inputs, outputs, initial_state, transitions):
        self.states = states
        self.inputs = inputs
        self.outputs = outputs
        self.initial_state = initial_state
        self.transitions = transitions

    def validate(self):
        if not self.states or not self.inputs or not self.outputs:
            raise ValueError("States, inputs, and outputs cannot be empty.")
        if self.initial_state not in self.states:
            raise ValueError(f"Initial state '{self.initial_state}' is not in states list.")
        for state in self.states:
            if state not in self.transitions:
                raise ValueError(f"Missing transitions for state '{state}'.")
            for inp in self.inputs:
                if inp not in self.transitions[state]:
                    raise ValueError(f"Missing transition for state '{state}' on input '{inp}'.")
                t = self.transitions[state][inp]
                if t["next_state"] not in self.states:
                    raise ValueError(f"Invalid next state '{t['next_state']}'.")
                if t["output"] not in self.outputs:
                    raise ValueError(f"Invalid output '{t['output']}'.")

    def convert(self):
        self.validate()

        incoming_outputs = defaultdict(set)
        for state in self.states:
            for inp in self.inputs:
                t = self.transitions[state][inp]
                incoming_outputs[t["next_state"]].add(t["output"])

        default_output = self.outputs[0]
        if not incoming_outputs[self.initial_state]:
            incoming_outputs[self.initial_state].add(default_output)

        moore_state_pairs = []
        for state in self.states:
            outs = sorted(list(incoming_outputs[state]))
            if not outs:
                outs = [default_output]
            for out in outs:
                moore_state_pairs.append((state, out))

        def format_name(state, out):
            if len(incoming_outputs[state]) > 1:
                return f"{state}_{out}"
            return state

        init_outs = sorted(list(incoming_outputs[self.initial_state]))
        chosen_init_out = default_output if default_output in init_outs else init_outs[0]
        moore_initial_state = format_name(self.initial_state, chosen_init_out)

        moore_states = []
        moore_outputs = {}
        moore_transitions = {}

        for orig_state, associated_out in moore_state_pairs:
            m_state_name = format_name(orig_state, associated_out)
            moore_states.append(m_state_name)
            moore_outputs[m_state_name] = associated_out
            moore_transitions[m_state_name] = {}

            for inp in self.inputs:
                mealy_t = self.transitions[orig_state][inp]
                target_orig = mealy_t["next_state"]
                target_out = mealy_t["output"]
                target_moore = format_name(target_orig, target_out)
                moore_transitions[m_state_name][inp] = target_moore

        return {
            "mealy": {
                "states": self.states,
                "inputs": self.inputs,
                "outputs": self.outputs,
                "initial_state": self.initial_state,
                "transitions": self.transitions,
                "dot": self._generate_mealy_dot()
            },
            "moore": {
                "states": moore_states,
                "inputs": self.inputs,
                "outputs": self.outputs,
                "initial_state": moore_initial_state,
                "state_outputs": moore_outputs,
                "transitions": moore_transitions,
                "dot": self._generate_moore_dot(
                    moore_states, moore_initial_state, moore_outputs, moore_transitions
                )
            }
        }

    def _generate_mealy_dot(self):
        lines = [
            'digraph Mealy {',
            '  rankdir=LR;',
            '  node [shape=circle, style=filled, fillcolor="#eef2ff", color="#4f46e5", fontname="Inter, sans-serif"];',
            '  edge [fontname="Inter, sans-serif", fontsize=11];',
            '  __start [shape=none, label="", width=0, height=0];',
            f'  __start -> "{self.initial_state}";'
        ]
        edge_labels = defaultdict(list)
        for state in self.states:
            for inp in self.inputs:
                t = self.transitions[state][inp]
                edge_labels[(state, t["next_state"])].append(f"{inp}/{t['output']}")

        for (src, dst), labels in edge_labels.items():
            label_str = ", ".join(labels)
            lines.append(f'  "{src}" -> "{dst}" [label="{label_str}"];')

        lines.append('}')
        return "\n".join(lines)

    def _generate_moore_dot(self, states, init_state, state_outputs, transitions):
        lines = [
            'digraph Moore {',
            '  rankdir=LR;',
            '  node [shape=circle, style=filled, fillcolor="#ecfdf5", color="#059669", fontname="Inter, sans-serif"];',
            '  edge [fontname="Inter, sans-serif", fontsize=11];',
            '  __start [shape=none, label="", width=0, height=0];',
            f'  __start -> "{init_state}";'
        ]
        for st in states:
            lines.append(f'  "{st}" [label="{st}\\n/ {state_outputs[st]}"];')

        edge_labels = defaultdict(list)
        for st in states:
            for inp in self.inputs:
                dst = transitions[st][inp]
                edge_labels[(st, dst)].append(inp)

        for (src, dst), labels in edge_labels.items():
            label_str = ", ".join(labels)
            lines.append(f'  "{src}" -> "{dst}" [label="{label_str}"];')

        lines.append('}')
        return "\n".join(lines)


@app.route("/")
def index():
    return render_template("index.html")


@app.route("/api/convert", methods=["POST"])
def convert_fsm():
    try:
        data = request.get_json()
        converter = MealyToMooreConverter(
            states=data["states"],
            inputs=data["inputs"],
            outputs=data["outputs"],
            initial_state=data["initial_state"],
            transitions=data["transitions"]
        )
        result = converter.convert()
        return jsonify({"success": True, "data": result})
    except Exception as e:
        return jsonify({"success": False, "error": str(e)}), 400


@app.route("/api/simulate", methods=["POST"])
def simulate_both():
    try:
        data = request.get_json()
        input_seq = data.get("sequence", [])
        mealy = data["mealy"]
        moore = data["moore"]

        curr_mealy = mealy["initial_state"]
        mealy_trace = []
        mealy_out_seq = []
        for sym in input_seq:
            if sym not in mealy["inputs"]:
                raise ValueError(f"Invalid input symbol '{sym}' in simulation sequence.")
            t = mealy["transitions"][curr_mealy][sym]
            mealy_trace.append({"from": curr_mealy, "input": sym, "to": t["next_state"], "output": t["output"]})
            mealy_out_seq.append(t["output"])
            curr_mealy = t["next_state"]

        curr_moore = moore["initial_state"]
        moore_initial_out = moore["state_outputs"][curr_moore]
        moore_trace = []
        moore_out_seq = []
        for sym in input_seq:
            nxt = moore["transitions"][curr_moore][sym]
            out = moore["state_outputs"][nxt]
            moore_trace.append({"from": curr_moore, "input": sym, "to": nxt, "output": out})
            moore_out_seq.append(out)
            curr_moore = nxt

        return jsonify({
            "success": True,
            "mealy_output": "".join(mealy_out_seq),
            "moore_initial_output": moore_initial_out,
            "moore_output": "".join(moore_out_seq),
            "equivalent": mealy_out_seq == moore_out_seq,
            "mealy_trace": mealy_trace,
            "moore_trace": moore_trace
        })
    except Exception as e:
        return jsonify({"success": False, "error": str(e)}), 400

Overwriting app.py


In [9]:
import threading
import time
from google.colab import output
from app import app

def run_server():
    app.run(host="0.0.0.0", port=5000, use_reloader=False)

threading.Thread(target=run_server, daemon=True).start()
time.sleep(1)

# Opens a clickable link / new browser tab directly to your running web app
output.serve_kernel_port_as_window(5000)

 * Serving Flask app 'app'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>